In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False


# 학습 모델 저장을 위한 라이브러리
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dfE=pd.read_csv('/content/drive/MyDrive/은서님 파일/model3_E,notE.csv')
dfE

,ID,Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,Not_E
3,TEST_00003,E
4,TEST_00004,E
...,...,...
99995,TEST_99995,E
99996,TEST_99996,E
99997,TEST_99997,E
99998,TEST_99998,Not_E


In [4]:
dfnotAB=pd.read_csv('/content/drive/MyDrive/은서님 파일/model3_A,B,notAB.csv')
dfnotAB

,ID,Segment
0,TEST_00002,notAB
1,TEST_00010,notAB
2,TEST_00012,notAB
3,TEST_00016,notAB
4,TEST_00032,notAB
...,...,...
16078,TEST_99957,notAB
16079,TEST_99961,notAB
16080,TEST_99982,notAB
16081,TEST_99994,notAB


In [5]:
dfCD=pd.read_csv('/content/drive/MyDrive/은서님 파일/model8459_C,D.csv')
dfCD

,ID,Segment
0,TEST_00002,D
1,TEST_00010,D
2,TEST_00012,D
3,TEST_00016,D
4,TEST_00032,D
...,...,...
16063,TEST_99957,D
16064,TEST_99961,C
16065,TEST_99982,C
16066,TEST_99994,D


In [6]:
# 1. dfCD에서 ID와 진짜 Segment만 추출
ab_mapping = dfnotAB[['ID', 'Segment']]

# 2. dfE와 ID 기준으로 병합 (진짜 Segment 값을 붙이기 위해)
dfE_updated = dfE.merge(ab_mapping, on='ID', how='left', suffixes=('', '_true'))

# 3. Not_E인 경우에만 진짜 Segment(C/D)로 교체
dfE_updated['Segment'] = dfE_updated.apply(
    lambda row: row['Segment_true'] if row['Segment'] == 'Not_E' and pd.notnull(row['Segment_true']) else row['Segment'],
    axis=1
)

# 4. 보조 컬럼 제거
dfE_updated = dfE_updated.drop(columns='Segment_true')


In [7]:
dfE_updated['Segment'].value_counts()

,count
Segment,
E,83917
notAB,16068
A,14
B,1


In [8]:
# 1. dfCD에서 ID와 진짜 Segment만 추출
cd_mapping = dfCD[['ID', 'Segment']]

# 2. dfE와 ID 기준으로 병합 (진짜 Segment 값을 붙이기 위해)
dfE_updated2 = dfE_updated.merge(cd_mapping, on='ID', how='left', suffixes=('', '_true'))

# 3. Not_E인 경우에만 진짜 Segment(C/D)로 교체
dfE_updated2['Segment'] = dfE_updated2.apply(
    lambda row: row['Segment_true'] if row['Segment'] == 'notAB' and pd.notnull(row['Segment_true']) else row['Segment'],
    axis=1
)

# 4. 보조 컬럼 제거
dfE_updated2 = dfE_updated2.drop(columns='Segment_true')


In [9]:
dfE_updated2['Segment'].value_counts()

,count
Segment,
E,83917
D,11582
C,4486
A,14
B,1


In [10]:
dfE_updated2

,ID,Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,D
3,TEST_00003,E
4,TEST_00004,E
...,...,...
99995,TEST_99995,E
99996,TEST_99996,E
99997,TEST_99997,E
99998,TEST_99998,C


In [11]:
dfE_updated2.to_csv('/content/drive/MyDrive/은서님 파일/result_model3(8459).csv',index=False, encoding='utf-8-sig')